# UNDERTONE - `gemma3n_e4b`

**google/gemma-3n-E4B-it** &nbsp;|&nbsp; 7.85B raw &nbsp;|&nbsp; ~16 GB fp16 &nbsp;|&nbsp; 2xT4
&nbsp;|&nbsp; documented ceiling **30 s** &nbsp;|&nbsp; primary scorer **logits**

[model card](https://huggingface.co/google/gemma-3n-E4B-it)

## What this model's documentation says

- **Gated.** Same licence step as E2B.
- Same 30 s ceiling; 7.85B raw params, so it shards over both T4s.

## What is shared with the other twelve notebooks

Prompt construction, per-run option shuffling, ladder windows (L1/L2/L3/L4),
letter-log-likelihood scoring, checkpointing and the output schema all come from
the `undertone` package. Only the adapter below is model-specific. That is the
whole design: tailored where the model demands it, identical everywhere else, so
the numbers compare.

## Protocol

Four options per item carrying **roles**, not fixed letters (letters are redrawn
per item per run, so preferring "A" cannot beat chance):

| role | content | diagnoses |
|---|---|---|
| `correct` | the right answer | - |
| `salience` | a louder / stressed / repeated competing mention | **salience prior** |
| `recency` | the most recent mention of the topic | recency bias |
| `absent` | "not mentioned in the recording" | fabrication; correct on null items |

Primary metric is the next-token distribution over A/B/C/D -- one forward pass,
no generation, no regex. Free generation runs alongside and `unparseable` is its
own bucket, never a wrong answer.

Cells whose window exceeds the ceiling above run **truncated and flagged**, and
are excluded from the accuracy table. They are never scored as zero.


In [ ]:
# Pinned for this model. If `load()` fails, this cell is the first thing to change.
%pip install -q "transformers==4.57.1"
%pip install -q "accelerate>=1.0.0"
%pip install -q "librosa>=0.10.2"
%pip install -q "soundfile>=0.12.1"
print("--- resolved versions (freeze these before the paper run) ---")
import importlib.metadata as md
for pkg in ["transformers", "accelerate", "torch", "librosa"]:
    try:
        print(f"{pkg:14s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:14s} not installed")

In [ ]:
import os, random, sys, json
import numpy as np, torch

SEED = 20260904
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Weights go to /kaggle/temp: scratch, and it does NOT count against the 20 GB
# /kaggle/working output cap. A 16-18 GB checkpoint in /kaggle/working would
# fail the commit at the end of the session.
os.environ.setdefault("HF_HOME", "/kaggle/temp/hf")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
# Long audio prompts allocate in large irregular blocks; without this the T4
# fragments and OOMs with a gigabyte nominally free.
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

# Gated model. The token is resolved after the repo is cloned (next cell), from:
#   HF_TOKEN in the environment -> Kaggle secret named HF_TOKEN -> .hf_token at
#   the repo root (gitignored - the repo is public, so a token in tracked source
#   would be scraped from GitHub within minutes).
# Accept the licence on the Hub with the account that owns the token first.
GATED = True

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"cuda:{i}  {p.name}  {p.total_memory/1e9:.1f} GB  sm{p.major}{p.minor}")
if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] < 8:
    print("\nsm < 80: no bf16 compute and no flash-attention-2. "
          "Every adapter loads in fp16 for this reason.")

In [ ]:
REPO_URL = "https://github.com/DeepanIsCool/longaudiobench.git"
REPO_REF = "undertone"   # pin to a commit sha before the paper run

import subprocess, shutil, os, sys
if os.path.exists("/kaggle/working/longaudiobench"):
    shutil.rmtree("/kaggle/working/longaudiobench")
for attempt in range(3):
    rc = subprocess.call(["git", "clone", "--depth", "1", "--branch", REPO_REF,
                          REPO_URL, "/kaggle/working/longaudiobench"])
    if rc == 0:
        break
else:
    raise RuntimeError("could not clone the benchmark repo")

sys.path.insert(0, "/kaggle/working/longaudiobench")
import importlib; importlib.invalidate_caches()

from undertone import ItemPack, adapters, env, runner, scoring
print("adapters registered:", len(adapters.list_adapters()))

if env.export_hf_token():
    print("HF token resolved")
elif globals().get("GATED"):
    raise RuntimeError(
        "this model is gated and no token was found. Add a Kaggle secret named "
        "HF_TOKEN, or write the token to .hf_token at the repo root.")

hw = env.resolve_hardware()
print(f"hardware: {hw.detail}  dtype={hw.dtype}  signature={hw.signature}")
print(f"versions: {env.versions()}")
# Every result row is stamped with this signature. The analysis refuses to put
# two signatures in one table -- a benchmark whose rows came from different
# backends compares machines, not models.

In [ ]:
# The adapter is the ONLY per-model code in this project. Everything above it --
# prompts, option shuffling, ladder windows, scoring, checkpointing -- is shared,
# which is what lets thirteen tailored notebooks produce comparable numbers.
import inspect
from undertone.adapters.base import get_adapter

ADAPTER_KEY = "gemma3n_e4b"
adapter = get_adapter(ADAPTER_KEY)

print(json.dumps(adapter.describe(), indent=2))
print("\n" + "=" * 72 + "\n")
print(inspect.getsource(type(adapter)))

In [ ]:
# Smoke-check this model in ITS OWN environment, before spending a sweep on it.
# The standalone 00_smoke_test loads every adapter in one env, which cannot work
# for a roster that needs three different transformers pins - so each model
# checks itself here, where its own pin is installed.
from undertone.smoke import smoke_adapter

report = smoke_adapter(adapter)
for name, result in report["checks"].items():
    print(f"  {'PASS' if result['ok'] else 'FAIL'}  {name}: {result['detail']}")
if report.get("traceback"):
    print(report["traceback"])

assert report.get("ok"), (
    f"{ADAPTER_KEY} failed: {report['failures']}. Fix the adapter before "
    "spending quota on a sweep - a broken adapter produces a table of zeros "
    "that looks like a finding.")
print(f"\nOK - peak {report.get('peak_vram_gb')} GB in {report.get('seconds')}s")

In [ ]:
# The item pack is built once on CPU (notebook 01) and attached as a Kaggle
# Dataset, so a model sweep never re-harvests audio.
# Locate the pack rather than assume the mount name. Kaggle derives the input
# directory from the dataset slug, and hardcoding "/kaggle/input/undertone-item-pack" failed
# twice against a dataset that was correctly attached the whole time.
import glob

candidates = sorted(glob.glob("/kaggle/input/*/item_pack.jsonl")
                    + glob.glob("/kaggle/working/item_pack/item_pack.jsonl"))
if not candidates:
    listing = sorted(glob.glob("/kaggle/input/*")) or ["(nothing mounted)"]
    raise FileNotFoundError(
        "no item_pack.jsonl under /kaggle/input. Attach the 'undertone-item-pack' dataset "
        f"(Add Input -> Datasets), or run 01_build_item_pack first. "
        f"Currently mounted: {listing}")
PACK_DIR = os.path.dirname(candidates[0])
print(f"item pack: {PACK_DIR}")

pack = ItemPack.load(os.path.join(PACK_DIR, "item_pack.jsonl"))
print(f"{len(pack)} items from {len({i.recording_id for i in pack})} recordings")
for key, n in sorted(pack.counts("lang", "category").items()):
    print(f"  {key[0]}  {key[1]}  n={n}")

# What this model can and cannot ingest, stated before the run rather than
# discovered from a table of zeros afterwards.
from undertone.ladder import CONDITIONS, window_for
print(f"\ndocumented ceiling: {adapter.max_audio_s:.0f} s")
for cond in CONDITIONS:
    over = sum(1 for i in pack if window_for(i, cond).seconds > adapter.max_audio_s)
    print(f"  {cond}: {over}/{len(pack)} cells exceed it -> truncated, not scored as 0")

In [ ]:
OUT = f"/kaggle/working/results/{{ADAPTER_KEY}}.jsonl"

# Resumable: a killed 12 h session picks up where it stopped. Rerun this cell
# after a restart rather than starting the sweep over.
runner.run_model(
    adapter,
    pack,
    out_path=OUT,
    conditions=CONDITIONS,
    seed=SEED,
    run_id="pilot",
    audio_root=PACK_DIR,
)
adapter.unload()
print("done ->", OUT)

In [ ]:
rows = runner.load_rows(OUT)
usable = runner.scorable(rows)          # drops errors and truncated cells
print(f"{{len(rows)}} rows, {{len(usable)}} scorable, "
      f"{{sum(1 for r in rows if r.get('truncated'))}} truncated, "
      f"{{sum(1 for r in rows if r.get('error'))}} errored")

by_cond = {{c: [r for r in usable if r["condition"] == c] for c in CONDITIONS}}
costs = scoring.ladder_costs(by_cond)
print("\nladder:", json.dumps({{k: round(v, 3) for k, v in costs.items()}}, indent=2))

print("\nper category (L3), salience trap is the headline diagnostic:")
for cat in ["P1", "P2", "P3", "P4", "C1"]:
    subset = [r for r in by_cond["L3"] if r["category"] == cat]
    if subset:
        s = scoring.summarize(subset)
        print(f"  {{cat}}  n={{s['n']}}  acc={{s['accuracy']:.3f}}  "
              f"salience={{s['salience_trap_rate']:.3f}}  "
              f"recency={{s['recency_trap_rate']:.3f}}")

degenerate = [r for r in usable if r.get("logit_degenerate")]
if degenerate:
    print(f"\nWARNING: {{len(degenerate)}} cells had all four letter logits equal. "
          "That is not a 25% baseline, it is a broken measurement -- check the adapter.")

with open(f"/kaggle/working/results/{{ADAPTER_KEY}}_summary.json", "w") as fh:
    json.dump({{"model": adapter.describe(), "ladder": costs,
               "overall": scoring.summarize(usable)}}, fh, indent=2, default=str)

import subprocess
subprocess.run(["tar", "-czf", f"/kaggle/working/{{ADAPTER_KEY}}_results.tar.gz",
                "-C", "/kaggle/working", "results"], check=True)
print("packaged")